In [1]:
import ROOT
import math
# import pandas as pd

# print(f"ROOT version: {ROOT.__version__}")
import numpy as np
from scipy.special import j0
import plotly.graph_objects as go
from scipy.integrate import fixed_quad


In [2]:
def read_data_file(filename):
    """Read data file and return arrays for x, y, y_error"""
    x_vals = []
    y_vals = []
    y_errs = []
    
    with open(filename, 'r') as f:
        for line in f:
            if line.strip() and not line.startswith('#'):
                parts = line.split()
                if len(parts) >= 3:
                    x_vals.append(float(parts[0]))
                    y_vals.append(float(parts[1]))
                    y_errs.append(float(parts[2]))
    
    return x_vals, y_vals, y_errs

In [3]:
# Load experimental data for all energies from ATLAS
x_atlas_all, y_atlas_all, yerr_atlas_all = read_data_file('../../../data/ens_atlas_difc0_2.dat')

# Function to process data for each energy block
def process_data(x_data, y_data, yerr_data, energy_blocks):
    x_values = []
    y_values = []
    y_errors = []
    
    for start, end in energy_blocks:
        if end is None:
            end = len(x_data)
        x_values.append(x_data[start:end])
        y_values.append(y_data[start:end])
        y_errors.append(yerr_data[start:end])
    
    return x_values, y_values, y_errors


In [4]:
#ranges for each energy 
atlas_blocks = [(0, 29), (29, 58), (58, None)]

# Process data
x_atlas, y_atlas, yerr_atlas = process_data(x_atlas_all, y_atlas_all, yerr_atlas_all, atlas_blocks)

# Extract values by energy
x_7_atlas, y_7_atlas, yerr_7_atlas = x_atlas[0], y_atlas[0], yerr_atlas[0]
x_8_atlas, y_8_atlas, yerr_8_atlas = x_atlas[1], y_atlas[1], yerr_atlas[1]
x_13_atlas, y_13_atlas, yerr_13_atlas = x_atlas[2], y_atlas[2], yerr_atlas[2]

In [5]:
# defining parameters/constants
b_0 = (33 - 6) / (12 * np.pi)
lambda_qcd = 0.284  # ΛQCD in GeV
gamma_1 = 0.084
gamma_2 = 2.36
rho = 4.0
s0 = 1.0
alpha_prime = 0.25


#ensemble parameters
param_mg_atlas_pl = 0.421
param_eps_atlas_pl = 0.0753
param_a1_atlas_pl = 1.517
param_a2_atlas_pl = 2.05

In [6]:
#--------------------------------------
# Eq 22 - GE
#--------------------------------------
def m2_pl(q2, mg):
    lambda2 = lambda_qcd ** 2
    rho_mg_2 = rho * (mg ** 2)
    ratio = math.log((q2 + rho_mg_2) / lambda2) / math.log(rho_mg_2 / lambda2)
    return (mg ** 4 / (q2 + mg ** 2)) * ratio ** (gamma_2 - 1)

#--------------------------------------
# Eq 26 - GE
#--------------------------------------
def G_p(q2, a1, a2):
    t = -q2
    return np.exp(-(a1 * np.abs(t) + a2 * np.abs(t) ** 2))


#--------------------------------------
# Eq 24 - GE
#--------------------------------------
def alpha_D(q2, mg, m2_type):
    m2_func = m2_type(q2, mg)
    return 1.0 / (b_0 * (q2 + m2_func) * math.log((q2 + 4 * m2_func) / (lambda_qcd ** 2)))

#--------------------------------------
# Eq 7 - GE
#--------------------------------------
def T_1(k, phi, mg, a1, a2, m2_type, q):
    q2 = q ** 2
    qk_cos = q * k * math.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2
    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_type)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_type)
    G0 = G_p(q, a1, a2)
    return alpha_D_plus * alpha_D_minus * G0 ** 2


#--------------------------------------
# Eq 8 - GE
#--------------------------------------
def T_2(k, phi, mg, a1, a2, m2_type, q):
    q2 = q ** 2
    qk_cos = q * k * math.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2
    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_type)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_type)
    factor = q2 + 9 * abs(k ** 2 - q2 / 4)
    G0 = G_p(q2, a1, a2)
    G_minus = G_p(factor, a1, a2)
    return alpha_D_plus * alpha_D_minus * G_minus * (2 * G0 - G_minus)


#--------------------------------------
# Eq 11 - GE
#--------------------------------------
def born_sigma_tot(amp_value, s):
    return amp_value.imag / s * 0.389379323

#--------------------------------------
# Eq 6 - GE
#--------------------------------------
def born_amp(diff_T, s, eps, t):
    alpha_pomeron = 1.0 + eps + 0.25 * t
    s_tilde = s/s0

    return 1j * s * 8 * (s_tilde**(alpha_pomeron - 1)) * diff_T

In [7]:
def root_1d_integrator(func, lower_limit, upper_limit):
    
    """
    PURPOSE: Perform one-dimensional numerical integration using ROOT's 
    adaptive integration method.

    PARAMETERS:

        func (callable): Function to be integrated. Must accept a single 
            floating-point argument.

        lower_limit (float): Lower bound of the integration interval.

        upper_limit (float): Upper bound of the integration interval.

    RETURNS:
        tuple: Tuple containing:
            - [0] (float): Estimated value of the integral.
            - [1] (float): Estimated uncertainty of the integral.
    """

    # creating Functor to be used by ROOT's integrator
    functor = ROOT.Math.Functor1D(func)

    type = ROOT.Math.IntegrationOneDim.kADAPTIVE   # integration type
    absTol = 1e-4
    relTol = 1e-4
    size   = 20
    rule   = ROOT.Math.Integration.kGAUSS15   
        
    # defining Integrator object
    integrator = ROOT.Math.IntegratorOneDim(type, absTol, relTol, size, rule)
    integrator.SetFunction(functor)
    
    # calculating integral and error 
    result = integrator.Integral(lower_limit, upper_limit)
    error = integrator.Error()
    
    return result, error


In [8]:
#--------------------------------------
# Eq 7 and Eq 8 - GE
#--------------------------------------

def k_integral(k, mg, a1, a2, m2_func, q):
    """
    PURPOSE: Define the integrand function over φ (phi) for a fixed k value.

    PARAMETERS:

        k (float): Momentum magnitude variable for the outer integral.

        mg (float): Effective gluon mass parameter.

        a1 (float): Model parameter a₁ related to the form factor.

        a2 (float): Model parameter a₂ related to the form factor.

        m2_func (callable): Function returning the squared mass term m²(q²) as a function 
            of momentum transfer.

        q (float): Momentum transfer variable in the scattering process.

    RETURNS:
        callable: Integrand function of φ (phi), defined as:
            f(φ) = k * [T₁(k, φ, mg, a1, a2, m2_func, q) - T₂(k, φ, mg, a1, a2, m2_func, q)].
    """
    integrand = lambda phi: k * (T_1(k, phi, mg, a1, a2, m2_func, q) -
                                 T_2(k, phi, mg, a1, a2, m2_func, q))
    return integrand


def phi_integral(phi, mg, a1, a2, m2_func, q, k_max):
    """
    PURPOSE: Compute the inner integral over k for a fixed φ (phi) value 
    using ROOT's 1D integrator.

    PARAMETERS:

        phi (float): Azimuthal angle variable (in radians).

        mg (float): Effective gluon mass parameter.

        a1 (float): Model parameter a₁ related to the form factor.

        a2 (float): Model parameter a₂ related to the form factor.

        m2_func (callable): Function returning the squared mass term m²(q²) as a function 
            of momentum transfer.

        q (float): Momentum transfer variable in the scattering process.

        k_max (float): Upper limit for the k integration.

    RETURNS:
        float: Estimated value of the k-integral for the given φ (phi):
            ∫₀^{k_max} k [T₁(k, φ) - T₂(k, φ)] dk.
    """
    def inner_in_k(k):
        return k * (T_1(k, phi, mg, a1, a2, m2_func, q) -
                    T_2(k, phi, mg, a1, a2, m2_func, q))
    
    result, _ = root_1d_integrator(inner_in_k, 0, k_max)
    return result


def compute_k_phi_integral(mg, a1, a2, m2_func, q, k_max):
    """
    PURPOSE: Compute the full two-dimensional integral over k and φ (phi),
    corresponding to Eqs. (7) and (8) in the GE model.

    PARAMETERS:

        mg (float): Effective gluon mass parameter.

        a1 (float): Model parameter a₁ related to the form factor.

        a2 (float): Model parameter a₂ related to the form factor.

        m2_func (callable): Function returning the squared mass term m²(q²) as a function 
            of momentum transfer.

        q (float): Momentum transfer variable in the scattering process.

        k_max (float): Upper limit for the k integration.

    RETURNS:
        tuple: Tuple containing:
            - [0] (float): Estimated value of the double integral:
                ∫₀^{2π} ∫₀^{k_max} k [T₁(k, φ) - T₂(k, φ)] dk dφ
            - [1] (float): Estimated uncertainty from the outer φ integration.
    """
    result, error = root_1d_integrator(
        lambda phi: phi_integral(phi, mg, a1, a2, m2_func, q, k_max),
        0,
        2 * math.pi
    )
    return result, error



# TESTING 

mg = param_mg_atlas_pl
a1 = param_a1_atlas_pl
a2 = param_a2_atlas_pl
q = 0
k_max = 13000

compute_k_phi_integral(mg, a1, a2, m2_pl, q, k_max)

(7.965987352559131, 8.844022572683941e-14)

In [9]:
#--------------------------------------  
#   COMPUTING SIGMA TOT BORN
#--------------------------------------


#start parameters
start_sqrt_s = 100
max_sqrt_s = 13000
step_size = 100

mg = param_mg_atlas_pl
a1 = param_a1_atlas_pl
a2 = param_a2_atlas_pl
q = 0

lst_sigma_tot_born_list = []
lst_sqrt_s_list = []

# start initial sqrt for while loop  
current_sqrt_s = start_sqrt_s

while current_sqrt_s <= max_sqrt_s:

    current_s = current_sqrt_s**2

    # calculates diff t (eq 7 and 8)
    diff_t,_ = compute_k_phi_integral(mg, a1, a2, m2_pl, q, current_sqrt_s)

    # calculating born amplitude
    born_amp_value = born_amp(diff_t, current_s, param_eps_atlas_pl, 0)
    # print(born_amp_value)

    #calculating sigma tot born
    born_sigma_tot_value = born_sigma_tot(born_amp_value, current_s)
    # print(born_sigma_tot_value)

    # append results to list
    lst_sigma_tot_born_list.append(born_sigma_tot_value)
    lst_sqrt_s_list.append(current_sqrt_s)

    # increase step
    current_sqrt_s += step_size 
    print(born_sigma_tot_value)

49.64805917299905
55.11090719675652
58.581020011925
61.17482154747304
63.26556087305723
65.02675168578644
66.55401305502859
67.90595095131731
69.12122172394115
70.22673606047499
71.24201962839247
72.18171228784657
73.05708730869492
73.877019663512
74.648627715497
75.37771259086793
76.06906749987411
76.72670079720794
77.35400020360356
77.95385597042524
78.52875475197625
79.08085220148368
79.61202984815023
80.12394018595793
80.61804280094172
81.09563360102334
81.55786867688803
82.0057839402058
82.44031140889837
82.86229280657722
83.27249099301083
83.67159962957827
84.0602513953254
84.4390250208399
84.80845131485145
85.16901837809974
85.5211761175423
85.86534017623246
86.20189536702067
86.53119868373891
86.85358195103674
87.16935416390376
87.47880355967614
87.78219945857967
88.07979390330043
88.37182312347103
88.65850884713969
88.94005947809886
89.21667115528243
89.48852870819165
89.75580652041585
90.01866931170441
90.27727284768216
90.53176458512979
90.78228425975604
91.02896442253143
91

In [10]:
#--------------------------------------
# Eq 23 - EIK
#--------------------------------------


def chi_eikonal(s, b, eps, mg, a1, a2, m2_func, born_amp_func):
    """
    PURPOSE: Compute the complex eikonal function χ(s, b) as defined in Eq. (23),
    using the Born amplitude and the nested integral over k and φ.

    PARAMETERS:

        s (float): Mandelstam variable s (squared center-of-mass energy).

        b (float): Impact parameter in femtometers (fm) or GeV⁻¹, depending on model units.

        eps (float): Model parameter ε controlling energy dependence of the amplitude.

        mg (float): Effective gluon mass parameter.

        a1 (float): Model parameter a₁ related to the form factor.

        a2 (float): Model parameter a₂ related to the form factor.

        m2_func (callable): Function returning the squared mass term m²(q²) as a function 
            of momentum transfer q.

        born_amp_func (callable): Function that computes the complex Born amplitude:
            A_Born(diff_T, s, eps, t), where t = -q² and diff_T is obtained from the 
            nested integral over k and φ.

    RETURNS:
        complex: Complex value of the eikonal function χ(s, b), computed as:
            (1 / s) × [∫ Re(A_Born) dq  +  i ∫ Im(A_Born) dq].

    NOTES:

        - Implements Eq. (23) from the generalized eikonal (GE) formalism.
        - Uses `root_1d_integrator` for numerical evaluation of the q-integral.
        - The integration limits (0 → 0.2) are set empirically and may depend 
          on the energy range or chosen model normalization.
    """

    def integrand_real(q):
        t = -q**2  
        
        # Compute diff_T for this q value
        diff_T, _ = compute_k_phi_integral(
            mg=mg,
            a1=a1,
            a2=a2,
            m2_func=m2_func,
            q=q,
            k_max=np.sqrt(s)
        )

        # Compute Born amplitude
        amp_born = born_amp_func(diff_T, s, eps, t)
        
        # Integrand for the real part
        return q * j0(b * q) * amp_born.real
    
    def integrand_imag(q):
        t = -q**2
        
        # Compute diff_T for this q value
        diff_T, _ = compute_k_phi_integral(
            mg=mg,
            a1=a1,
            a2=a2,
            m2_func=m2_func,
            q=q,
            k_max=np.sqrt(s)
        )

        # Compute Born amplitude
        amp_born = born_amp_func(diff_T, s, eps, t)
        
        # Integrand for the imaginary part
        return q * j0(b * q) * amp_born.imag
    
    # Perform numerical integration for real and imaginary components
    real_integral_result, _ = root_1d_integrator(integrand_real, 0.0, 0.2)
    imag_integral_result, _ = root_1d_integrator(integrand_imag, 0.0, 0.2)

    # Combine real and imaginary parts
    integral_result = real_integral_result + 1j * imag_integral_result

    return integral_result / s


print(chi_eikonal(7000**2, 10.0, param_eps_atlas_pl, param_mg_atlas_pl, param_a1_atlas_pl, param_a2_atlas_pl,m2_pl, born_amp))

1.511339210070323j


In [11]:
#--------------------------------------
# Eq 24 - EIK (using the new chi_eikonal)
#--------------------------------------

def eik_amp(s, t, eps, mg, a1, a2, m2_func, born_amp_func, q_max=0.2):
    """
    PURPOSE: Compute the eikonalized scattering amplitude A_eik(s, t) 
    as defined in Eq. (24) using the eikonal function χ(s, b).

    PARAMETERS:

        s (float): Mandelstam variable s (squared center-of-mass energy).

        t (float): Mandelstam variable t (momentum transfer squared, typically negative).

        eps (float): Model parameter ε controlling the energy dependence of the amplitude.

        mg (float): Effective gluon mass parameter.

        a1 (float): Model parameter a₁ related to the form factor.

        a2 (float): Model parameter a₂ related to the form factor.

        m2_func (callable): Function returning the squared mass term m²(q²) as a function 
            of momentum transfer.

        born_amp_func (callable): Function that computes the complex Born amplitude:
            A_Born(diff_T, s, eps, t), where diff_T is obtained from the nested integral 
            over k and φ.

        q_max (float, optional): Maximum momentum transfer q used internally in χ(s, b) 
            integration. Default is 0.2.

    RETURNS:
        complex: Complex eikonalized amplitude A_eik(s, t), given by:
            i s × ∫₀^{b_max} b J₀(b√(-t)) [1 - exp(iχ(s, b))] db.

    NOTES:

        - Implements Eq. (24) from the generalized eikonal (GE) formalism.
        - Uses the `chi_eikonal` function to evaluate χ(s, b) at each b value.
        - The integration is performed over b ∈ [0, 30], which may be adjusted 
          depending on the physical range or model normalization.
        - `root_1d_integrator` is used for adaptive numerical integration.
    """

    q = np.sqrt(-t)  # q = √(-t) since t = -q²
    
    def integrand_real(b_val):
        # Compute eikonal function χ(s, b)
        chi_val = chi_eikonal(s, b_val, eps, mg, a1, a2, m2_func, born_amp_func)
        
        # Compute [1 - exp(iχ(s, b))]
        exp_term = np.exp(1j * chi_val)
        one_minus_exp = 1.0 - exp_term
        
        # Integrand: b * J₀(b√(-t)) * [1 - exp(iχ(s, b))]
        return (b_val * j0(b_val * q) * one_minus_exp).real
    
    def integrand_imag(b_val):
        # Compute eikonal function χ(s, b)
        chi_val = chi_eikonal(s, b_val, eps, mg, a1, a2, m2_func, born_amp_func)
        
        # Compute [1 - exp(iχ(s, b))]
        exp_term = np.exp(1j * chi_val)
        one_minus_exp = 1.0 - exp_term
        
        # Integrand: b * J₀(b√(-t)) * [1 - exp(iχ(s, b))]
        return (b_val * j0(b_val * q) * one_minus_exp).imag
    
    # Integrate real and imaginary parts separately
    real_integral, _ = root_1d_integrator(integrand_real, 0.0, 30)
    imag_integral, _ = root_1d_integrator(integrand_imag, 0.0, 30)
    
    # Combine real and imaginary components
    integral_result = real_integral + 1j * imag_integral
    
    # Final amplitude: i s times the integral
    return 1j * s * integral_result

print(eik_amp(7000**2,-0.04,param_eps_atlas_pl,param_mg_atlas_pl,param_a1_atlas_pl,param_a2_atlas_pl,m2_pl, born_amp))

369946026.85824126j


In [12]:
import ROOT
import numpy as np
import math


def root_minimize(func,
                  ndim,
                  minimizerName="Minuit2",
                  algoName="",
                  mg_init=None,
                  eps_init=None,
                  a1_init=None,
                  a2_init=None,
                  stepSize=None,
                  maxFunctionCalls=1000000,
                  maxIterations=10000,
                  tolerance=1e-8,
                  printLevel=0):
    """
    PURPOSE: Generic wrapper to perform parameter minimization using ROOT's 
    built-in minimizers, returning parameter values, Hesse and MINOS errors.

    PARAMETERS:

        func (callable): Function to minimize. Should accept a list or numpy array 
            of length `ndim`.

        ndim (int): Number of parameters to minimize.

        minimizerName (str, optional): Minimizer to use (Minuit, Minuit2, 
            GSLMultiMin, GSLSimAn, Genetic, etc.). Default is "Minuit2".

        algoName (str, optional): Specific algorithm to use (Migrad, BFGS, 
            ConjugateFR, Simplex, etc.). Default is "".

        mg_init (float or None, optional): Initial guess for parameter 'mg'. 
            Defaults to 0.0 if None.

        eps_init (float or None, optional): Initial guess for parameter 'eps'. 
            Defaults to 0.0 if None.

        a1_init (float or None, optional): Initial guess for parameter 'a1'. 
            Defaults to 0.0 if None.

        a2_init (float or None, optional): Initial guess for parameter 'a2'. 
            Defaults to 0.0 if None.

        stepSize (list of floats or None, optional): Step sizes for each parameter. 
            Defaults to 0.01 for all parameters.

        maxFunctionCalls (int, optional): Maximum allowed function evaluations. 
            Default is 1,000,000.

        maxIterations (int, optional): Maximum allowed iterations. Default is 10,000.

        tolerance (float, optional): Desired convergence tolerance. Default is 1e-8.

        printLevel (int, optional): Verbosity of the minimizer (0=quiet, 1=normal, 
            2=verbose). Default is 1.

    RETURNS:
        dict: Dictionary containing minimization results:
            - 'success' (bool): Whether minimization converged successfully.
            - 'x' (numpy.ndarray): Parameter values at minimum.
            - 'status' (int): Minimizer status (0 = success).
            - 'hesse_errors' (numpy.ndarray): Symmetric Hesse errors.
            - 'minos_errors_low' (numpy.ndarray): Lower MINOS errors.
            - 'minos_errors_up' (numpy.ndarray): Upper MINOS errors.
    """
    
    #-------------------
    #  SET STARTING POINT
    #-------------------

    param_names = ["mg", "eps", "a1", "a2"]

    startPoint = [
        mg_init if mg_init is not None else 0.0,
        eps_init if eps_init is not None else 0.0,
        a1_init if a1_init is not None else 0.0,
        a2_init if a2_init is not None else 0.0
    ][:ndim]  # garante que só use ndim parâmetros

    # OLD VERSION OF THE CODE ABOVE
    # init_map = [mg_init, eps_init, a1_init, a2_init]
    # startPoint = []
    # for i in range(ndim):
    #     if init_map[i] is not None:
    #         startPoint.append(init_map[i])
    #     else:
    #         startPoint.append(0.0)  # fallback default

    
    # --------------------------------------------------------------

    #-------------------
    #  SET STEP SIZE
    #-------------------
    if stepSize is None:
        stepSize = [0.01] * ndim

    #-------------------
    #  SET CONFIDENCE LEVEL FOR 4D CASE 90% CL
    #-------------------
    errordef = 7.78
    
    #-------------------
    #  CREATE MINIMIZER
    #-------------------

    minimizer = ROOT.Math.Factory.CreateMinimizer(minimizerName, algoName)
    if not minimizer:
        raise RuntimeError(f"Cannot create minimizer \"{minimizerName}\"")
    

    #-------------------
    #  SET OPTIONS
    #-------------------

    minimizer.SetMaxFunctionCalls(maxFunctionCalls)
    minimizer.SetMaxIterations(maxIterations)
    minimizer.SetTolerance(tolerance)
    minimizer.SetPrintLevel(printLevel)
    minimizer.SetErrorDef(errordef)
    f = ROOT.Math.Functor(func, ndim)
    minimizer.SetFunction(f)

    
    variable = list(startPoint)

    #-------------------
    #  SET PARAMETERS
    #-------------------

    # renaming variable names to match model parameters and set parameters 
    if ndim >= 1:
        minimizer.SetVariable(0, param_names[0], variable[0], stepSize[0])
    if ndim >= 2:
        minimizer.SetVariable(1, param_names[1], variable[1], stepSize[1])
    if ndim >= 3:
        minimizer.SetVariable(2, param_names[2], variable[2], stepSize[2])
    if ndim >= 4:
        minimizer.SetVariable(3, param_names[3], variable[3], stepSize[3])

    # OLD VERSION OF THE CODE ABOVE
    # for i in range(ndim):
    #     if i < len(param_names):
    #         name = param_names[i]
    #     else:
    #         name = f"x{i}"
    #     minimizer.SetVariable(i, name, variable[i], stepSize[i])

    

    
    """
    could replace the code above by the following code to set parameters without renaming
    for i in range(ndim):
        minimizer.SetVariable(i, f"x{i}", variable[i], stepSize[i])
    """

    #-------------------
    #  RUN MINIMIZATION
    #-------------------

    minimization = minimizer.Minimize()
    if not minimization:
        return {'success': False}
    

    #-------------------
    # GET HESSE ERROR
    #-------------------

    # Create empty arrays to store the results
    xs = np.zeros(ndim)           # parameter values at minimum
    hesse_errors = np.zeros(ndim) # symmetric Hesse errors

    # Loop over each parameter and extract the value and Hesse error
    if ndim >= 1:
        xs[0] = minimizer.X()[0]
        hesse_errors[0] = minimizer.Errors()[0]
    if ndim >= 2:
        xs[1] = minimizer.X()[1]
        hesse_errors[1] = minimizer.Errors()[1]
    if ndim >= 3:
        xs[2] = minimizer.X()[2]
        hesse_errors[2] = minimizer.Errors()[2]
    if ndim >= 4:
        xs[3] = minimizer.X()[3]
        hesse_errors[3] = minimizer.Errors()[3]

    # OLD VERSION OF THE CODE ABOVE
    # for i in range(ndim):
    #     xs[i] = minimizer.X()[i]          # get the fitted value of parameter i
    #     hesse_errors[i] = minimizer.Errors()[i]  # get the Hesse error for parameter i


    #-------------------
    # GET MINOS ERROR
    #-------------------
    
    # Initialize arrays to store MINOS errors
    minos_errors_low = np.zeros(ndim)
    minos_errors_up = np.zeros(ndim)

    # Temporary arrays for ROOT's GetMinosError
    errLow = np.zeros(1, dtype=np.float64)
    errUp  = np.zeros(1, dtype=np.float64)

    if ndim >= 1:
        success = minimizer.GetMinosError(0, errLow, errUp)
        if success:
            minos_errors_low[0] = errLow[0]
            minos_errors_up[0] = errUp[0]
        else:
            minos_errors_low[0] = -hesse_errors[0]
            minos_errors_up[0] = hesse_errors[0]

    if ndim >= 2:
        success = minimizer.GetMinosError(1, errLow, errUp)
        if success:
            minos_errors_low[1] = errLow[0]
            minos_errors_up[1] = errUp[0]
        else:
            minos_errors_low[1] = -hesse_errors[1]
            minos_errors_up[1] = hesse_errors[1]

    if ndim >= 3:
        success = minimizer.GetMinosError(2, errLow, errUp)
        if success:
            minos_errors_low[2] = errLow[0]
            minos_errors_up[2] = errUp[0]
        else:
            minos_errors_low[2] = -hesse_errors[2]
            minos_errors_up[2] = hesse_errors[2]

    if ndim >= 4:
        success = minimizer.GetMinosError(3, errLow, errUp)
        if success:
            minos_errors_low[3] = errLow[0]
            minos_errors_up[3] = errUp[0]
        else:
            minos_errors_low[3] = -hesse_errors[3]
            minos_errors_up[3] = hesse_errors[3]

    # OLD VERSION OF THE CODE ABOVE
    # for i in range(ndim):
    #     success = minimizer.GetMinosError(i, errLow, errUp)
    #     if success:
    #         minos_errors_low[i] = errLow[0]
    #         minos_errors_up[i] = errUp[0]
    #     else:
    #         # fallback to Hesse errors if MINOS fails
    #         minos_errors_low[i] = -hesse_errors[i]
    #         minos_errors_up[i] = hesse_errors[i]



    # print results
    print("\nMinimization results (values ± Hesse ± MINOS):")
    for i in range(ndim):
        print(f"{param_names[i]}: {xs[i]:.6f} "
              f"± {hesse_errors[i]:.6f} "
              f"[{minos_errors_low[i]:+.6f}, {minos_errors_up[i]:+.6f}]")

    print(f"\nStatus: {minimizer.Status()} (0 = success)\n")
    # ----------------------

    return {
        'success': minimization and minimizer.Status() == 0,
        'x': xs,
        'status': minimizer.Status(),
        'hesse_errors': hesse_errors,
        'minos_errors_low': minos_errors_low,
        'minos_errors_up': minos_errors_up,
    }

# TESTING
def StyblinskiTang4D(vecx):
    result = 0.0
    for i in range(4):
        x = vecx[i]
        result += x**4 - 16*x**2 + 5*x
    return result / 2.0

result = root_minimize(StyblinskiTang4D, ndim=4, mg_init=-2.5, eps_init=-2.5, a1_init=-2.5, a2_init=-2.5, printLevel=0)


Minimization results (values ± Hesse ± MINOS):
mg: -2.903534 ± 0.670769 [-0.608517, +0.770186]
eps: -2.903534 ± 0.670769 [-0.608517, +0.770186]
a1: -2.903534 ± 0.670769 [-0.608517, +0.770186]
a2: -2.903534 ± 0.670769 [-0.608517, +0.770186]

Status: 0 (0 = success)



In [13]:
def get_chi2_minimization(func_model, x_data, y_data, y_errors, initial_params, param_limits, 
             xmin, xmax, root_minimize, minimizerName="Minuit2", algoName="Migrad", 
             **minimize_options):
    """
    PURPOSE: Function to perform chi-squared minimization using ROOT's built-in methods.

    PARAMETERS:

        func_model (str): Model function to fit.

        x_data (array): x-values of the data points.

        y_data (array): y-values of the data points.

        y_errors (array): uncertainties on y-values.

        initial_params (array): initial guess for the parameters.

        param_limits (list of tuples, optional): parameter limits [(min, max), ...].

        xmin (float): minimum x-value for the fit range.

        xmax (float): maximum x-value for the fit range.

        root_minimize (function): ROOT's minimization function (e.g., ROOT.minimize).

        minimizerName (str, optional): minimizer to use (default is "Minuit2").

        algoName (str, optional): minimization algorithm (default is "Migrad").

        **minimize_options: additional options for root_minimize.
    
    RETURNS:
        dict: best-fit parameters and fit results.
    """

    # Number of data points
    n_points = len(x_data)
    
    # parameter names for output
    param_names = ['mg', 'eps', 'a1', 'a2']

    # number of parameters
    npar = len(initial_params)
    
    # creates a ROOT TGraphErrors object with the data points
    graph = ROOT.TGraphErrors(n_points)
    for i in range(n_points):
        graph.SetPoint(i, x_data[i], y_data[i])
        graph.SetPointError(i, 0, y_errors[i])
    
    # creates a ROOT TF1 object with the model function
    func = ROOT.TF1("fit_func", func_model, xmin, xmax, npar)
    
    # renaming variables to match model parameters
    func.SetParName(0, 'mg')
    func.SetParName(1, 'eps')
    func.SetParName(2, 'a1')
    func.SetParName(3, 'a2')
    
    # set initial parameter values
    func.SetParameter(0, initial_params[0])
    func.SetParameter(1, initial_params[1])
    func.SetParameter(2, initial_params[2])
    func.SetParameter(3, initial_params[3])
    
    # Definir limites individualmente (se existirem)
    # if param_limits and len(param_limits) >= 4:
    #     func.SetParLimits(0, param_limits[0][0], param_limits[0][1])
    #     func.SetParLimits(1, param_limits[1][0], param_limits[1][1])
    #     func.SetParLimits(2, param_limits[2][0], param_limits[2][1])
    #     func.SetParLimits(3, param_limits[3][0], param_limits[3][1])
    
    # Define chi-square function using built in ROOT function
    def chi2_func(params):
        func.SetParameter(0, params[0])
        func.SetParameter(1, params[1])
        func.SetParameter(2, params[2])
        func.SetParameter(3, params[3])
        return graph.Chisquare(func)
    
    # get root_minimize arguments
    minimize_args = {
        'func': chi2_func,
        'ndim': npar,
        'minimizerName': minimizerName,
        'algoName': algoName,
        'mg_init': initial_params[0],
        'eps_init': initial_params[1],
        'a1_init': initial_params[2],
        'a2_init': initial_params[3],
    }
    minimize_args.update(minimize_options)
    
    # execute minimization
    result = root_minimize(**minimize_args)
    
    if result['success']:
        # update parameter values in the model function
        func.SetParameter(0, result['x'][0])
        func.SetParameter(1, result['x'][1])
        func.SetParameter(2, result['x'][2])
        func.SetParameter(3, result['x'][3])
        
        # calculate chi-square and degrees of freedom   
        chi2 = graph.Chisquare(func)
        ndf = n_points - npar
        
        # output dictionary
        output = {
            'chi2': chi2,
            'ndf': ndf,
            'chi2_dof': chi2 / ndf if ndf > 0 else 0.0,
            'parameters': result['x'].tolist(),
            'errors': result['hesse_errors'].tolist(),
            'minos_errors_low': result['minos_errors_low'].tolist(),
            'minos_errors_up': result['minos_errors_up'].tolist(),
            'param_names': param_names,
            'valid': True,
            'status': result['status'],
            'function': func,
            'covariance_matrix': result.get('covariance_matrix', None)
        }

    # if fail minimization
    else:
        output = {
            'chi2': 0.0,
            'ndf': 0,
            'chi2_dof': 0.0,
            'parameters': initial_params,
            'errors': [0.0] * npar,
            'minos_errors_low': [0.0] * npar,
            'minos_errors_up': [0.0] * npar,
            'param_names': param_names,
            'valid': False,
            'status': -1,
            'function': func,
            'covariance_matrix': None
        }
    
    return output

# FUNCTION TEST BELOW

In [22]:
import numpy as np
import ROOT

def test_chi2_minimization_different_model():
    """
    Test chi-squared minimization with a different 4-parameter model function.
    """
    
    # 1. Define a different model function - Lorentzian peak with linear background
    func_model = "[0] / ((x - [1])*(x - [1]) + [2]) + [3]*x"
    # Parameters: [amplitude, center, width, slope]
    
    # 2. True parameter values
    true_params = [5.0, 2.0, 0.5, 0.2]  # [amplitude, center, width, slope]
    
    # 3. Generate synthetic data
    n_points = 100
    x_min, x_max = 0, 4
    x_data = np.linspace(x_min, x_max, n_points)
    
    # Calculate true y values without noise (Lorentzian + linear background)
    y_true = (true_params[0] / ((x_data - true_params[1])**2 + true_params[2]) + 
              true_params[3] * x_data)
    
    # 4. Add realistic Gaussian noise
    noise_level = 0.05  # 5% noise
    y_errors = np.abs(y_true) * noise_level + 0.1  # minimum error of 0.1
    y_data = y_true + np.random.normal(0, y_errors)
    
    # 5. Set up minimization inputs
    initial_params = [4.0, 1.8, 0.6, 0.15]  # Slightly different from true values
    param_limits = None
    
    # 6. Call the minimization function
    result = get_chi2_minimization(
        func_model=func_model,
        x_data=x_data,
        y_data=y_data,
        y_errors=y_errors,
        initial_params=initial_params,
        param_limits=param_limits,
        xmin=x_min,
        xmax=x_max,
        root_minimize=root_minimize
    )
    
    # 7. Display results
    print("=" * 60)
    print("CHI-SQUARED MINIMIZATION TEST - LORENTZIAN + LINEAR BACKGROUND")
    print("=" * 60)
    print(f"Fit successful: {result['valid']}")
    print(f"Chi2/ndf: {result['chi2_dof']:.3f}")
    print(f"Model: amplitude / ((x - center)² + width) + slope*x")
    print()
    
    param_names = ['amplitude', 'center', 'width', 'slope']
    print("Parameter comparison:")
    print("Parameter  | True Value | Fitted Value | Difference | Error")
    print("-" * 65)
    
    for i, name in enumerate(param_names):
        true_val = true_params[i]
        fitted_val = result['parameters'][i]
        error = result['errors'][i]
        diff = fitted_val - true_val
        
        print(f"{name:10} | {true_val:10.4f} | {fitted_val:12.4f} | {diff:9.4f} | {error:5.4f}")
    
    print()
    
    # Check if fitted parameters are close to true values (within 3 sigma)
    all_close = True
    for i in range(len(true_params)):
        if abs(result['parameters'][i] - true_params[i]) > 3 * result['errors'][i]:
            all_close = False
            print(f"  Parameter {param_names[i]} differs by {abs(result['parameters'][i] - true_params[i])/result['errors'][i]:.1f} sigma")
    
    if all_close and result['valid']:
        print("✓ TEST PASSED: Fitted parameters consistent with true values")
    else:
        print("✗ TEST FAILED: Parameters not recovered correctly")
    
    return result

# Additional test with exponential decay model
def test_chi2_minimization_exponential():
    """
    Test with exponential decay model with constant offset.
    """
    
    # 1. Define exponential decay model
    func_model = "[0] * exp(-[1]*x) + [2] * exp(-[3]*x)"
    # Parameters: [amp1, tau1, amp2, tau2]
    
    # 2. True parameter values - double exponential
    true_params = [3.0, 1.5, 2.0, 0.3]  # [amp1, tau1, amp2, tau2]
    
    # 3. Generate synthetic data
    n_points = 80
    x_min, x_max = 0, 6
    x_data = np.linspace(x_min, x_max, n_points)
    
    # Calculate true y values (double exponential decay)
    y_true = (true_params[0] * np.exp(-true_params[1] * x_data) + 
              true_params[2] * np.exp(-true_params[3] * x_data))
    
    # 4. Add realistic Gaussian noise
    noise_level = 0.08  # 8% noise
    y_errors = np.abs(y_true) * noise_level + 0.05  # minimum error
    y_data = y_true + np.random.normal(0, y_errors)
    
    # 5. Set up minimization inputs
    initial_params = [2.5, 1.2, 1.8, 0.4]  # Slightly different guesses
    param_limits = None
    
    # 6. Call the minimization function
    result = get_chi2_minimization(
        func_model=func_model,
        x_data=x_data,
        y_data=y_data,
        y_errors=y_errors,
        initial_params=initial_params,
        param_limits=param_limits,
        xmin=x_min,
        xmax=x_max,
        root_minimize=root_minimize
    )
    
    # 7. Display results
    print("\n" + "=" * 60)
    print("CHI-SQUARED MINIMIZATION TEST - DOUBLE EXPONENTIAL DECAY")
    print("=" * 60)
    print(f"Fit successful: {result['valid']}")
    print(f"Chi2/ndf: {result['chi2_dof']:.3f}")
    print(f"Model: amp1 * exp(-tau1*x) + amp2 * exp(-tau2*x)")
    print()
    
    param_names = ['amp1', 'tau1', 'amp2', 'tau2']
    print("Parameter comparison:")
    print("Parameter | True Value | Fitted Value | Difference | Error")
    print("-" * 65)
    
    for i, name in enumerate(param_names):
        true_val = true_params[i]
        fitted_val = result['parameters'][i]
        error = result['errors'][i]
        diff = fitted_val - true_val
        
        print(f"{name:9} | {true_val:10.4f} | {fitted_val:12.4f} | {diff:9.4f} | {error:5.4f}")
    
    print()
    
    # Check if fitted parameters are close to true values
    all_close = True
    for i in range(len(true_params)):
        if abs(result['parameters'][i] - true_params[i]) > 3 * result['errors'][i]:
            all_close = False
            print(f"  Parameter {param_names[i]} differs by {abs(result['parameters'][i] - true_params[i])/result['errors'][i]:.1f} sigma")
    
    if all_close and result['valid']:
        print("✓ TEST PASSED: Fitted parameters consistent with true values")
    else:
        print("✗ TEST FAILED: Parameters not recovered correctly")
    
    return result

# Run all tests
if __name__ == "__main__":
    print("Testing chi-squared minimization with different model functions...\n")
    
    # Test 1: Lorentzian + linear background
    result1 = test_chi2_minimization_different_model()
    
    # Test 2: Double exponential decay  
    # result2 = test_chi2_minimization_exponential()
    
    print("\n" + "=" * 60)
    print("SUMMARY")
    print("=" * 60)
    print(f"Test 1 (Lorentzian): {'PASS' if result1['valid'] and result1['chi2_dof'] < 3 else 'FAIL'}")
    # print(f"Test 2 (Exponential): {'PASS' if result2['valid'] and result2['chi2_dof'] < 3 else 'FAIL'}")

Testing chi-squared minimization with different model functions...


Minimization results (values ± Hesse ± MINOS):
mg: 4.670578 ± 0.374076 [-0.360322, +0.388857]
eps: 1.975134 ± 0.024455 [-0.024431, +0.024516]
a1: 0.465517 ± 0.045963 [-0.043721, +0.048395]
a2: 0.250073 ± 0.055692 [-0.056362, +0.055048]

Status: 0 (0 = success)

CHI-SQUARED MINIMIZATION TEST - LORENTZIAN + LINEAR BACKGROUND
Fit successful: True
Chi2/ndf: 1.119
Model: amplitude / ((x - center)² + width) + slope*x

Parameter comparison:
Parameter  | True Value | Fitted Value | Difference | Error
-----------------------------------------------------------------
amplitude  |     5.0000 |       4.6706 |   -0.3294 | 0.3741
center     |     2.0000 |       1.9751 |   -0.0249 | 0.0245
width      |     0.5000 |       0.4655 |   -0.0345 | 0.0460
slope      |     0.2000 |       0.2501 |    0.0501 | 0.0557

✓ TEST PASSED: Fitted parameters consistent with true values

SUMMARY
Test 1 (Lorentzian): PASS


In [15]:
# #--------------------------------------
# # Eq 23 - EIK
# #--------------------------------------

# q_max = 0.2

# def chi_eikonal(s, b, eps, mg, a1, a2, m2_func):
#     """
#     Eikonal function χ(s,b) from eq. (23)
#     χ(s,b) = (1/s) ∫ q dq J₀(bq) A_Born(s,t)
#     where t = -q²
#     """
#     def integrand_real(q):
#         t = -q**2  
        
#         # Calculate diff_T for this q value        
#         diff_T, _ = compute_k_phi_integral(
#             mg=mg,
#             a1=a1,
#             a2=a2,
#             m2_func=m2_func,
#             q=q,
#             k_max=np.sqrt(s)  # Use sqrt(s) as k_max
#         )

#         # Calculate Born amplitude for this t
#         amp_born = born_amp(diff_T, s, eps, t)
        
#         # Integrand: q * J₀(b*q) * A_Born(s,t)
#         return q * j0(b * q) * amp_born.real
    
#     def integrand_imag(q):
#         t = -q**2  # t = -q² as specified
        
#         # Calculate diff_T for this q value        
#         diff_T, _ = compute_k_phi_integral(
#             mg=mg,
#             a1=a1,
#             a2=a2,
#             m2_func=m2_func,
#             q=q,
#             k_max=np.sqrt(s)  # Use sqrt(s) as k_max
#         )

#         # Calculate Born amplitude for this t
#         amp_born = born_amp(diff_T, s, eps, t)
        
#         # Integrand: q * J₀(b*q) * A_Born(s,t)
#         return q * j0(b * q) * amp_born.imag
    
#     real_integral_result, _ = root_1d_integrator(integrand_real, 0.0, q_max)  
#     imag_integral_result, _ = root_1d_integrator(integrand_imag, 0.0, q_max)  

#     integral_result = real_integral_result + 1j*imag_integral_result

#     return integral_result / s

# print(chi_eikonal(7000**2, 10.0, param_eps_atlas_pl, param_mg_atlas_pl, param_a1_atlas_pl, param_a2_atlas_pl,m2_pl))

# #--------------------------------------
# # Eq 24 - EIK
# #--------------------------------------

# b_max = 30  # Maximum impact parameter

# def eikonal_amplitude(s, t, eps, mg, a1, a2, m2_func):
#     """
#     Eikonalized amplitude from eq. (24)
#     A_eik(s,t) = i s ∫ b db J₀(b√(-t)) [1 - exp(iχ(s,b))]
#     where t is the Mandelstam variable (negative)
#     """
#     q = np.sqrt(-t)  # q = √(-t) since t = -q²
    
#     def integrand_real(b):
#         # Calculate eikonal function χ(s,b)
#         chi_val = chi_eikonal(s, b, eps, mg, a1, a2, m2_func)
        
#         # Compute [1 - exp(iχ(s,b))]
#         exp_term = np.exp(1j * chi_val)
#         one_minus_exp = 1.0 - exp_term
        
#         # Integrand: b * J₀(b√(-t)) * [1 - exp(iχ(s,b))]
#         return (b * j0(b * 0) * one_minus_exp).real
    
#     def integrand_imag(b):
#         # Calculate eikonal function χ(s,b)
#         chi_val = chi_eikonal(s, b, eps, mg, a1, a2, m2_func)
        
#         # Compute [1 - exp(iχ(s,b))]
#         exp_term = np.exp(1j * chi_val)
#         one_minus_exp = 1.0 - exp_term
        
#         # Integrand: b * J₀(b√(-t)) * [1 - exp(iχ(s,b))]
#         return (b * j0(b * 0) * one_minus_exp).imag
    
#     # Integrate real and imaginary parts separately
#     real_integral, _ = root_1d_integrator(integrand_real, 0.0, b_max)
#     imag_integral, _ = root_1d_integrator(integrand_imag, 0.0, b_max)
    
#     integral_result = real_integral + 1j * imag_integral
    
#     # Final amplitude: i s times the integral
#     return 1j * s * integral_result

# print(eikonal_amplitude(7000**2,-0.04,param_eps_atlas_pl,param_mg_atlas_pl,param_a1_atlas_pl,param_a2_atlas_pl,m2_pl))